In [1]:
import pandas as pd
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.models.hhem_model import load_hhem_model, get_hhem_score
from src.data.prepare_data import prepare_hhem_data


# Load prepared dataset
df_hhem_validate, df_hhem_test = prepare_hhem_data()

context_validate = df_hhem_validate.iloc[:100]["passages"]
response_validate = df_hhem_validate.iloc[:100]["response"]


## Test data
context_test = df_hhem_test.iloc[:100]["passages"]
response_test = df_hhem_test.iloc[:100]["response"]

In [2]:
model = load_hhem_model()

# scores =[]
# for context , response in zip(context_validate,response_validate):
#     score = get_hhem_score(model, context, response)
#     scores.append(score)

# get scores for 
scores = get_hhem_score(model, context_validate, response_validate)

You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.
2026-10-28 13:50:16.450097: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Token indices sequence length is longer than the specified maximum sequence length for this model (1034 > 512). Running this sequence through the model will result in indexing errors


In [87]:
from pprint import pprint
df_hhem_validate = df_hhem_validate.iloc[:100].copy()
df_hhem_validate["hhem_score"] = scores
df_hhem_validate
label_score = df_hhem_validate[["hallucination_label", "hhem_score", "source_id"]].copy()
label_score["scored_label"] = label_score["hhem_score"].apply(lambda x: "GROUNDED" if x >= .90  else  "Grey Zone" if x > .40   else "HALLUCINATED")
mismatched = label_score[label_score["hallucination_label"] != label_score["scored_label"]]
print(len(mismatched))
mismatched["source_id"].value_counts()

38


source_id
14308    4
14295    4
14297    4
14301    3
14296    3
14299    3
14293    3
14302    2
14303    2
14304    2
14305    2
14306    1
14307    1
14292    1
14298    1
14294    1
14309    1
Name: count, dtype: int64

In [116]:


print(mismatched[mismatched["source_id"] == 14309 ])
# df_hhem_validate.loc[
#     [98, 99, 100, 101],
#     ["passages", "response", "hhem_score", "hallucination_label"]
# ]


for i in [105]:
    print("\nROW:", i)
    print("PASSAGES:\n", df_hhem_validate.loc[i, "passages"])
    print("\nRESPONSE:\n", df_hhem_validate.loc[i, "response"])
    print("\nHHEM SCORE:", df_hhem_validate.loc[i, "hhem_score"])
    print("TRUE LABEL:", df_hhem_validate.loc[i, "hallucination_label"])

    hallucination_label  hhem_score  source_id scored_label
105        HALLUCINATED    0.467172      14309    Grey Zone

ROW: 105
PASSAGES:
 passage 1:Remove prime rib from refrigerator and allow the meat to come to room temperature, about 3 hours depending on the size of the roast. Preheat oven to 425 degrees F (220 degrees C). Line a roasting pan with aluminum foil. Unwrap the roast and place onto roasting pan; blot with paper towels. Sift together flour, pepper, salt, paprika, onion powder, garlic powder, and celery seed into a small bowl. Coat the roast all over with the flour mixture. Roast in preheated oven until cooked to your desired degree of doneness; about 4 1/2 hours (20 minutes a pound) for medium-rare.

passage 2:To check the temperature of your oven, place an oven thermometer on the center rack and heat the oven for 15 minutes at 350 degrees F. If, after 15 minutes, the oven temperature reads higher or lower than your oven setting, adjust your oven setting for your prime